## Dashen Bank Review Analysis

Import Required Libraries

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

print('Path set. Python will now look in:', os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

Path set. Python will now look in: C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics


In [2]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime
from src.data_scrapper import scrape_metadata, scrape_reviews, data_quality_check
from src.data_preprocessor import missing_values, duplicate_reviews, check_dateFormat, missing_values, normalize_date, clean_text, invalid_reviews, remove_duplicates, save_cleaned_data, preprocessing_report

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


Defining Variables

In [3]:
App_Id = "com.dashen.dashensuperapp"
bank = 'Dashen Bank'
source = 'Google Play Store'
print(f" Scraping Review for Bank: {bank} with App ID: {App_Id} from Source: {source}")


 Scraping Review for Bank: Dashen Bank with App ID: com.dashen.dashensuperapp from Source: Google Play Store


## Mobile App Review Data Scraping from Google Play Store

In [4]:
# Get meta data for the app
scrape_metadata(App_Id, bank)

App Info for Dashen Bank
App Title   : Dashen Bank
Current Score: 4.242478
Total Ratings: 5,636
Total Reviews: 1,022
Installs     : 1,000,000+


{'title': 'Dashen Bank',
 'description': 'Why choose Dashen Bank Super App?\r\n• Secure & Reliable: Built with robust security to protect your financial transactions.\r\n• Convenient: Manage your accounts, pay bills, and access essential services in one place.\r\n• User-Friendly: Simplified design for seamless navigation and usage.',
 'descriptionHTML': 'Why choose Dashen Bank Super App?<br>• Secure &amp; Reliable: Built with robust security to protect your financial transactions.<br>• Convenient: Manage your accounts, pay bills, and access essential services in one place.<br>• User-Friendly: Simplified design for seamless navigation and usage.',
 'summary': 'Bank smarter, pay bills easily, and access essential lifestyle services',
 'installs': '1,000,000+',
 'minInstalls': 1000000,
 'realInstalls': 1526401,
 'score': 4.242478,
 'ratings': 5636,
 'reviews': 1022,
 'histogram': [608, 269, 299, 428, 4029],
 'price': 0,
 'free': True,
 'currency': 'USD',
 'sale': False,
 'saleTime': None,

In [5]:
#scrape 500 reviews for the app
df_reviews, continuation_token = scrape_reviews(App_Id, bank, count=500)
df_reviews = pd.DataFrame(df_reviews)

Collected 500 raw reviews for Dashen Bank app.


In [6]:
print(type(df_reviews))

<class 'pandas.DataFrame'>


In [7]:
print(df_reviews.head())
print(df_reviews.columns)

                               reviewId       userName  \
0  78284116-8313-4ca8-a28c-61ce4fa44ee6      king mele   
1  b4a7ea95-727b-4501-8d84-77db88187c84     Ahmed Abdi   
2  59a20f9e-bb87-4d5f-bee7-ce19f19baddf        Bellixs   
3  fc184115-ab13-482b-bb08-798589a4d482  biniyam yared   
4  105aa71e-1e72-4618-899b-78a321a5258b    Mohamed Ali   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a/ACg8oc...   
1  https://play-lh.googleusercontent.com/a/ACg8oc...   
2  https://play-lh.googleusercontent.com/a-/ALV-U...   
3  https://play-lh.googleusercontent.com/a-/ALV-U...   
4  https://play-lh.googleusercontent.com/a/ACg8oc...   

                                             content  score  thumbsUpCount  \
0  Very Annoying App i tried to open virtual bank...      1              0   
1                                               good      5              0   
2                                               good      5              0   
3 

In [8]:
# Inspect what a single raw review looks like

print("Keys in a single review:")
print(list(df_reviews.iloc[0].keys()))

print("\nFirst raw review (sample):")
for key, value in df_reviews.iloc[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 78284116-8313-4ca8-a28c-61ce4fa44ee6
  userName: king mele
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocJ-Sd3vVfFN4yivVIhKiK-JnVbFJFmZE_4tI8f1DNFOgOuWiQ=mo
  content: Very Annoying App i tried to open virtual bank account with fayda but in the end it says something went wrong i tried so many times it says something went wrong why???fix it quickly for now i give you 1/5👎👎👎
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 1.9.14
  at: 2026-05-14 12:33:42
  replyContent: nan
  repliedAt: NaT
  appVersion: 1.9.14


In [9]:
# Step 3: Extract only the columns we need
raw_data = []

for r in df_reviews.itertuples():
    raw_data.append({
        'review_id': r.reviewId,
        'review'   : r.content,
        'rating'   : r.score,
        'date'     : r.at,
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,78284116-8313-4ca8-a28c-61ce4fa44ee6,Very Annoying App i tried to open virtual bank...,1,2026-05-14 12:33:42,Awash Bank,Google Play
1,b4a7ea95-727b-4501-8d84-77db88187c84,good,5,2026-05-14 12:26:20,Awash Bank,Google Play
2,59a20f9e-bb87-4d5f-bee7-ce19f19baddf,good,5,2026-05-14 10:35:20,Awash Bank,Google Play
3,fc184115-ab13-482b-bb08-798589a4d482,good app but it was doesnt work other bank tra...,5,2026-05-14 10:29:26,Awash Bank,Google Play
4,105aa71e-1e72-4618-899b-78a321a5258b,good,5,2026-05-14 09:15:23,Awash Bank,Google Play


## Exploring the Raw Data

In [10]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [11]:
print("Rating Distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 4) 
    print(f"{rating} stars: {bar} ({count} reviews)")


Rating Distribution:
5 stars: ████████████████████████████████████████████████████████████████████████████████ (322 reviews)
4 stars: ████████ (33 reviews)
3 stars: ██████ (27 reviews)
2 stars: ████ (19 reviews)
1 stars: ████████████████████████ (99 reviews)


In [12]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-14 12:33:42
1   2026-05-14 12:26:20
2   2026-05-14 10:35:20
3   2026-05-14 10:29:26
4   2026-05-14 09:15:23
5   2026-05-14 08:58:59
6   2026-05-14 08:19:29
7   2026-05-13 12:14:11
8   2026-05-13 10:44:57
9   2026-05-13 09:26:31

Date dtype: datetime64[us]


## Data Quality Audit

In [13]:
data_quality_check(df_raw)

Data Quality Check:
------------------------------
Total reviews collected: 500
Missing values per column:
review_id    0
review       0
rating       0
date         0
bank         0
source       0
dtype: int64


In [14]:
missing_values(df_raw)

Removed 0 rows with missing critical data
Remaining: 500 reviews


,review_id,review,rating,date,bank,source
0,78284116-8313-4ca8-a28c-61ce4fa44ee6,Very Annoying App i tried to open virtual bank...,1,2026-05-14 12:33:42,Awash Bank,Google Play
1,b4a7ea95-727b-4501-8d84-77db88187c84,good,5,2026-05-14 12:26:20,Awash Bank,Google Play
2,59a20f9e-bb87-4d5f-bee7-ce19f19baddf,good,5,2026-05-14 10:35:20,Awash Bank,Google Play
3,fc184115-ab13-482b-bb08-798589a4d482,good app but it was doesnt work other bank tra...,5,2026-05-14 10:29:26,Awash Bank,Google Play
4,105aa71e-1e72-4618-899b-78a321a5258b,good,5,2026-05-14 09:15:23,Awash Bank,Google Play
...,...,...,...,...,...,...
495,35fc8b13-0a39-4237-b28d-39904b57f125,good,5,2025-08-14 09:14:19,Awash Bank,Google Play
496,5043e566-8cbc-4384-9382-8d929065614e,There are plenty of features missed e. g elect...,2,2025-08-14 08:16:24,Awash Bank,Google Play
497,0269ed2c-74ca-4b44-8226-0a664489ac78,የሶፍትዌሩ ለአጠቃቀም ምቹና ቀላል መሆኑ ተመራጭ ያደርገዋል,5,2025-08-14 06:56:25,Awash Bank,Google Play
498,9a553128-a2f5-4c25-8c0c-abc680033d42,በጣም ችግር አለበት,1,2025-08-14 05:45:15,Awash Bank,Google Play


Check for Duplicates

In [15]:
duplicate_reviews(df_raw)

Duplicate reviews:
------------------------------
Total duplicate review IDs: 0
Total duplicate review texts: 94
Total empty reviews: 0
Total duplicate reviews: 0


In [16]:
# Copying the raw DataFrame to work on a clean version
df = df_raw.copy()
print("Data copied for preprocessing.")

Data copied for preprocessing.


Remove Missing Data

In [17]:
df = missing_values(df)

Removed 0 rows with missing critical data
Remaining: 500 reviews


Remove Duplicates

In [18]:
df = remove_duplicates(df)

Removed 0 duplicate reviews based on review_id
Remaining: 500 reviews


Normalize Date

In [19]:
check_dateFormat(df)
df = normalize_date(df)


Checking date format:
------------------------------
Sample dates: 2026-05-14 12:33:42
Data type of 'date' column: datetime64[us]
  Target format: YYYY-MM-DD (string or date object)
Dates normalized to YYYY-MM-DD format
dtype: str

Date range: 2025-08-14 to 2026-05-14


Clean white spaces

In [20]:
df = clean_text(df)

Remove Invalid Reviews

In [21]:
df = invalid_reviews(df)

Invalid ratings (outside 1–5): 0
Remaining reviews after removing invalid ratings: 500
Data type of 'rating' column: int64


## Cleaned Data

In [22]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

TypeError: 'NoneType' object is not subscriptable

Save Cleaned Data

In [ ]:
df = save_cleaned_data(df, "cbe_reviews")

## Report for Data Preprocessing

In [ ]:
preprocessing_report(df_raw, df)